# AstroCLIMB — Qwen3-VL-8B three-seed ensemble + swap TTA

Inference-only Kaggle notebook for the full-vocabulary language-attention QLoRA adapters trained with seeds **17, 42, and 123**. For each adapter it averages predictions from `(obj_1, obj_2)` and `(obj_2, obj_1)`, then averages the three seed-level probability vectors.

Attach the AstroCLIMB competition data, the Qwen3-VL-8B base model (unless internet access is enabled), and all three saved adapter output datasets. Name each adapter dataset/path with `seed17`, `seed42`, or `seed123`, or fill in `ADAPTER_OVERRIDES` below. This experiment deliberately does **not** apply a modality mask or calibration. Enable a Kaggle **T4 x2** accelerator.

In [ ]:
# Preserve Kaggle's torch/torchvision stack.
%pip install -q --upgrade --upgrade-strategy only-if-needed "transformers==4.57.1" "peft==0.17.1" "accelerate==1.10.1" "bitsandbytes==0.47.0" "qwen-vl-utils==0.0.14"

In [ ]:
import base64
import csv
import hashlib
import io
import json
import math
import os
import re
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True
csv.field_size_limit(sys.maxsize)

MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
SEEDS = [17, 42, 123]
TARGET_COLUMNS = ['same_figure', 'same_paper', 'related_papers', 'unrelated_papers']
MIN_PIXELS = 256 * 256
MAX_PIXELS = 448 * 448
MAX_TEXT_CHARS = 3000
TEST_LIMIT = None  # Set to 100 for a smoke/timing test; None creates the full submission.
REBUILD_CACHE = False

# Fill these only if automatic discovery cannot identify an attached adapter.
ADAPTER_OVERRIDES = {17: None, 42: None, 123: None}

WORK_ROOT = Path('/kaggle/working/astroclimb_three_seed_swap_tta') if Path('/kaggle/working').exists() else Path('./astroclimb_three_seed_swap_tta')
IMAGE_ROOT = WORK_ROOT / 'images_448'
TEST_MANIFEST = WORK_ROOT / 'test_10000.jsonl'
for path in (WORK_ROOT, IMAGE_ROOT):
    path.mkdir(parents=True, exist_ok=True)

def locate_csv(filename):
    preferred = [Path('/kaggle/input/competitions/astroclimb') / filename, Path('/kaggle/input/astroclimb') / filename]
    for candidate in preferred:
        if candidate.exists():
            return candidate
    roots = [Path('/kaggle/input'), Path('data')]
    candidates = [p for root in roots if root.exists() for p in root.rglob(filename)]
    candidates = sorted(candidates, key=lambda p: ('astroclimb' not in str(p).lower(), len(str(p))))
    if not candidates:
        raise FileNotFoundError(f'{filename} not found')
    return candidates[0]

def locate_model():
    if Path('/kaggle/input').exists():
        candidates = []
        for config in Path('/kaggle/input').rglob('config.json'):
            text = str(config.parent).lower()
            if 'qwen3' in text and 'vl' in text and '8b' in text and not (config.parent / 'adapter_config.json').exists():
                candidates.append(config.parent)
        if candidates:
            return str(sorted(candidates, key=lambda p: len(str(p)))[0])
    return MODEL_ID

def locate_adapter(seed):
    override = ADAPTER_OVERRIDES[seed]
    if override:
        path = Path(override)
        if not (path / 'adapter_config.json').exists():
            raise FileNotFoundError(f'Invalid adapter override for seed {seed}: {path}')
        return path
    if not Path('/kaggle/input').exists():
        raise FileNotFoundError(f'Set ADAPTER_OVERRIDES[{seed}] when running outside Kaggle')
    all_adapters = [p.parent for p in Path('/kaggle/input').rglob('adapter_config.json')]
    pattern = re.compile(rf'(^|[^0-9])seed[-_]?{seed}([^0-9]|$)', re.IGNORECASE)
    matches = [p for p in all_adapters if pattern.search(str(p))]
    if len(matches) != 1:
        listing = '\n'.join(f'  - {p}' for p in all_adapters) or '  (none)'
        raise RuntimeError(f'Expected one adapter path containing seed{seed}; found {len(matches)}. Set ADAPTER_OVERRIDES[{seed}].\nAvailable adapters:\n{listing}')
    return matches[0]

TEST_CSV = locate_csv('test.csv')
MODEL_PATH = locate_model()
ADAPTER_PATHS = {seed: locate_adapter(seed) for seed in SEEDS}
print('Test:', TEST_CSV)
print('Model:', MODEL_PATH)
for seed, path in ADAPTER_PATHS.items():
    print(f'Seed {seed} adapter:', path)
print('Output:', WORK_ROOT)

## Cache test objects once

The three adapters reuse one manifest and one decoded/resized image cache.

In [ ]:
def looks_like_image(value):
    return isinstance(value, str) and value.lstrip().startswith(('iVBORw0KGgo', '/9j/', 'UklGR', 'R0lGOD', 'data:image'))

def decode_image(value):
    value = value.strip()
    if value.startswith('data:image'):
        value = value.split(',', 1)[1]
    image = Image.open(io.BytesIO(base64.b64decode(value, validate=False)))
    image.load()
    return image.convert('RGB')

def resize_to_area(image, max_pixels=MAX_PIXELS):
    width, height = image.size
    if width * height <= max_pixels:
        return image
    scale = math.sqrt(max_pixels / (width * height))
    return image.resize((max(1, round(width * scale)), max(1, round(height * scale))), Image.Resampling.LANCZOS)

def cache_object(value):
    if not looks_like_image(value):
        return {'kind': 'caption', 'value': value}
    digest = hashlib.sha256(value.encode('utf-8')).hexdigest()
    destination = IMAGE_ROOT / f'{digest}.png'
    if not destination.exists():
        resize_to_area(decode_image(value)).save(destination, format='PNG', compress_level=3)
    return {'kind': 'image', 'value': str(destination)}

def build_test_manifest():
    started = time.perf_counter()
    count = 0
    with TEST_CSV.open('r', encoding='utf-8', newline='') as source, TEST_MANIFEST.open('w', encoding='utf-8') as destination:
        reader = csv.DictReader(source)
        missing = {'id', 'obj_1', 'obj_2'} - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f'test.csv missing columns: {sorted(missing)}')
        for row in reader:
            record = {'id': row['id'], 'obj_1': cache_object(row['obj_1']), 'obj_2': cache_object(row['obj_2'])}
            destination.write(json.dumps(record, ensure_ascii=False) + '\n')
            count += 1
            if count % 500 == 0:
                print(f'Cached {count} rows in {(time.perf_counter() - started) / 60:.1f} min')
    return count

if REBUILD_CACHE or not TEST_MANIFEST.exists():
    test_count = build_test_manifest()
else:
    test_count = sum(1 for _ in TEST_MANIFEST.open(encoding='utf-8'))
    print('Reusing:', TEST_MANIFEST)
assert test_count == 10000
print('Rows:', test_count, '| cached images:', len(list(IMAGE_ROOT.glob('*.png'))))

## Write the isolated two-GPU swap-TTA inference worker

In [ ]:
%%writefile /kaggle/working/astroclimb_swap_tta_infer.py
import argparse
import csv
import json
import os
import time
from pathlib import Path

import torch
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True
MIN_PIXELS = 256 * 256
MAX_PIXELS = 448 * 448
MAX_TEXT_CHARS = 3000
TARGET_COLUMNS = ['same_figure', 'same_paper', 'related_papers', 'unrelated_papers']
SYSTEM_PROMPT = '''You classify the relationship between two objects from astronomy papers.
0: The objects are the figure and caption of the same scientific figure.
1: The objects are from different figures in the same paper.
2: The objects are from different papers and one paper cites the other.
3: The objects are from unrelated papers.
The relationship is symmetric. Output only one digit: 0, 1, 2, or 3.'''.strip()

def shorten_caption(text):
    if len(text) <= MAX_TEXT_CHARS:
        return text
    half = MAX_TEXT_CHARS // 2
    return text[:half] + '\n[...middle truncated...]\n' + text[-half:]

def object_content(number, obj):
    if obj['kind'] == 'image':
        with Image.open(obj['value']) as source:
            image = source.convert('RGB')
        return [{'type': 'text', 'text': f'Object {number} is a scientific figure:'}, {'type': 'image', 'image': image}]
    return [{'type': 'text', 'text': f'Object {number} is a figure caption:\n{shorten_caption(obj["value"])}'}]

def build_messages(row, swap=False):
    obj_1, obj_2 = row['obj_1'], row['obj_2']
    if swap:
        obj_1, obj_2 = obj_2, obj_1
    content = object_content(1, obj_1) + object_content(2, obj_2)
    content.append({'type': 'text', 'text': 'Classify their relationship. Reply with one digit only.'})
    return [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user', 'content': content},
    ]

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--test-manifest', required=True)
    parser.add_argument('--model-path', required=True)
    parser.add_argument('--adapter-dir', required=True)
    parser.add_argument('--output-dir', required=True)
    parser.add_argument('--test-limit', type=int, default=-1)
    args = parser.parse_args()

    rank = int(os.environ.get('LOCAL_RANK', '0'))
    world_size = int(os.environ.get('WORLD_SIZE', '2'))
    torch.cuda.set_device(rank)
    from peft import PeftModel
    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

    with Path(args.test_manifest).open('r', encoding='utf-8') as handle:
        rows = [json.loads(line) for line in handle]
    if args.test_limit >= 0:
        rows = rows[:args.test_limit]
    rows = rows[rank::world_size]

    processor = AutoProcessor.from_pretrained(args.adapter_dir, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    base = Qwen3VLForConditionalGeneration.from_pretrained(
        args.model_path, quantization_config=quantization, dtype=torch.float16,
        attn_implementation='sdpa', device_map={'': rank},
    )
    model = PeftModel.from_pretrained(base, args.adapter_dir)
    model.eval()
    model.config.use_cache = True
    token_ids = []
    for digit in '0123':
        ids = processor.tokenizer.encode(digit, add_special_tokens=False)
        if len(ids) != 1:
            raise ValueError(f'Label {digit} is not one token: {ids}')
        token_ids.append(ids[0])

    @torch.inference_mode()
    def predict(row, swap):
        batch = processor.apply_chat_template(
            build_messages(row, swap=swap), tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors='pt',
        )
        batch = {key: value.to(model.device) if torch.is_tensor(value) else value for key, value in batch.items()}
        logits = model(**batch).logits[0, -1, token_ids].float()
        return torch.softmax(logits, dim=-1).cpu().numpy()

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f'probabilities_rank{rank}.csv'
    started = time.perf_counter()
    with output_path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['id', *[f'p_{name}' for name in TARGET_COLUMNS]])
        writer.writeheader()
        for index, row in enumerate(rows, start=1):
            probabilities = 0.5 * (predict(row, swap=False) + predict(row, swap=True))
            writer.writerow({'id': row['id'], **{f'p_{name}': float(probabilities[i]) for i, name in enumerate(TARGET_COLUMNS)}})
            if index % 100 == 0:
                elapsed = time.perf_counter() - started
                print(f'rank={rank} {index}/{len(rows)} {elapsed/index:.3f}s/row ETA={(elapsed/index)*(len(rows)-index)/3600:.2f}h', flush=True)
    elapsed = time.perf_counter() - started
    print(f'Rank {rank} finished {len(rows)} rows in {elapsed/3600:.2f}h', flush=True)

if __name__ == '__main__':
    main()

## Run swap TTA for each seed

Each seed is processed sequentially; each process uses both GPUs by sharding rows. The model process exits between seeds, releasing GPU memory.

In [ ]:
INFERENCE_SCRIPT = Path('/kaggle/working/astroclimb_swap_tta_infer.py')
launch_env = dict(os.environ, PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
seed_probability_paths = {}

for seed in SEEDS:
    output_dir = WORK_ROOT / f'seed{seed}_swap_tta_shards'
    command = [
        sys.executable, '-m', 'accelerate.commands.launch', '--multi_gpu', '--num_processes', '2',
        str(INFERENCE_SCRIPT), '--test-manifest', str(TEST_MANIFEST), '--model-path', MODEL_PATH,
        '--adapter-dir', str(ADAPTER_PATHS[seed]), '--output-dir', str(output_dir),
        '--test-limit', str(-1 if TEST_LIMIT is None else TEST_LIMIT),
    ]
    print(f'Launching seed {seed}:', ' '.join(command), flush=True)
    started = time.perf_counter()
    subprocess.run(command, check=True, env=launch_env)
    print(f'Seed {seed} swap TTA finished in {(time.perf_counter()-started)/3600:.2f}h')

    shards = [pd.read_csv(output_dir / f'probabilities_rank{rank}.csv', dtype={'id': str}) for rank in range(2)]
    probabilities = pd.concat(shards, ignore_index=True)
    with TEST_MANIFEST.open('r', encoding='utf-8') as handle:
        ordered_ids = [json.loads(line)['id'] for line in handle]
    if TEST_LIMIT is not None:
        ordered_ids = ordered_ids[:TEST_LIMIT]
    order = {str(row_id): index for index, row_id in enumerate(ordered_ids)}
    probabilities['order'] = probabilities['id'].map(order)
    assert probabilities['order'].notna().all()
    probabilities = probabilities.sort_values('order').drop(columns='order').reset_index(drop=True)
    assert probabilities['id'].is_unique and probabilities['id'].tolist() == list(map(str, ordered_ids))
    merged_path = WORK_ROOT / f'seed{seed}_swap_tta_probabilities.csv'
    probabilities.to_csv(merged_path, index=False)
    seed_probability_paths[seed] = merged_path
    print('Saved:', merged_path)

## Equal-weight probability ensemble and submission validation

In [ ]:
probability_columns = [f'p_{name}' for name in TARGET_COLUMNS]
frames = {seed: pd.read_csv(path, dtype={'id': str}) for seed, path in seed_probability_paths.items()}
reference_ids = frames[SEEDS[0]]['id'].tolist()
for seed, frame in frames.items():
    assert frame.columns.tolist() == ['id', *probability_columns]
    assert frame['id'].is_unique and frame['id'].tolist() == reference_ids
    values = frame[probability_columns].to_numpy(dtype=np.float64)
    assert np.isfinite(values).all() and (values >= 0).all()
    assert np.allclose(values.sum(axis=1), 1.0, atol=1e-5)

ensemble_values = np.mean([frames[seed][probability_columns].to_numpy(dtype=np.float64) for seed in SEEDS], axis=0)
ensemble_values /= ensemble_values.sum(axis=1, keepdims=True)
ensemble_probabilities = pd.DataFrame(ensemble_values, columns=probability_columns)
ensemble_probabilities.insert(0, 'id', reference_ids)
predictions = ensemble_values.argmax(axis=1)
submission = pd.DataFrame({'id': reference_ids})
for class_index, name in enumerate(TARGET_COLUMNS):
    submission[name] = (predictions == class_index).astype(np.int8)

assert submission.columns.tolist() == ['id', *TARGET_COLUMNS]
assert submission['id'].is_unique
assert submission[TARGET_COLUMNS].isin([0, 1]).all().all()
assert (submission[TARGET_COLUMNS].sum(axis=1) == 1).all()
if TEST_LIMIT is None:
    assert len(submission) == 10000

probability_path = WORK_ROOT / 'submission_probabilities.csv'
submission_path = WORK_ROOT / 'submission.csv'
ensemble_probabilities.to_csv(probability_path, index=False)
submission.to_csv(submission_path, index=False)
print('Rows:', len(submission))
print('Prediction counts:', submission[TARGET_COLUMNS].sum().to_dict())
print('Probabilities:', probability_path)
print('Submission:', submission_path)
if TEST_LIMIT is not None:
    print('WARNING: partial timing output only; set TEST_LIMIT=None for a valid Kaggle submission.')
display(submission.head())